# core

> `rishi` registers seven backends with [Urai](https://vedicreader.github.io/urai) and provides helpers for its own code.

Urai defines the `Chat` class, the tool loop, callbacks, options, model capabilities and recorded replies. Urai does not run an inference engine. A backend is a `Chat` subclass that runs one inference engine. A runtime is the name that Urai assigns to a backend, such as `'llama'` or `'remote'`. Importing `rishi.core` registers the backends. `Chat('some-model')` then selects the matching backend.

The top-level `rishi` package re-exports Urai names. `from rishi import *` imports those names. `rishi.core` does not copy them.

In [ ]:
#| default_exp core

In [ ]:
#| export
import ast, asyncio, inspect, time, warnings
from fastcore.all import Path, first, find_file_parents
from urai import (Caps, Runtime, RUNTIMES, register_runtime, infer_runtime, resolve_runtime, resolve, model_caps, cfg_caps, mmproj_caps,
  is_path, split_runtime, use_system_certs, mk_content, mk_msg, mk_msgs, http_approval, ToolLoopMixin, SlidingWindowCallback,
  ToolReminderCallback, UsageCallback, UsageStats)

In [ ]:
#| export
class RishiToolLoop(ToolLoopMixin):
    "Base class for rishi backends that drive Urai's tool loop. The usage, tool-reminder and sliding-window callbacks are on by default."
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    oneshot_stream_ok = False
    oneshot_deadline_ok = False
    oneshot_max_tokens_ok = False
    oneshot_use = UsageStats()

def usage_stats(u):
    "One call's `UsageStats` from the usage dict of `norm_claude_usage` or `norm_completion`; empty when `u` is empty."
    if not u: return UsageStats()
    return UsageStats(u.get('prompt_tokens', 0), u.get('completion_tokens', 0), u.get('total_tokens', 0), 1, cached_tokens=u.get('cached_tokens', 0),
                      model=u.get('model'), reasoning_tokens=u.get('reasoning_tokens', 0), cache_creation_tokens=u.get('cache_creation_tokens', 0), cost=u.get('cost', 0.0))

class OneShotCancelled(Exception):
    "Raised when `cancel()` ends a `oneshot` or `oneshot_stream` call, or when its provider stream does not confirm the close. `elapsed` is the run time in seconds, read after the close. `stopped` is False when the provider call did not end within the wait, and that call can still finish and spend. rishi never retries the call. Text yielded before the error is a partial reply."
    def __init__(self, elapsed, stopped):
        self.elapsed, self.stopped = elapsed, stopped
        super().__init__(f'the one-shot was cancelled after {elapsed:.1f}s; ' +
                         ('its call was stopped' if stopped else 'its call did not stop in time, so it may still finish and spend'))

CANCEL_POLL, CLOSE_WAIT = 0.1, 20

async def read_or_cancel(task, cancel=None, timeout=None):
    "Wait for `task`, a timeout or a set `threading.Event`, then return whether `task` is done without cancelling it."
    if cancel is None: await asyncio.wait({task}, timeout=timeout); return task.done()
    end = None if timeout is None else time.monotonic() + timeout
    while not task.done() and not cancel.is_set():
        left = CANCEL_POLL if end is None else min(CANCEL_POLL, end - time.monotonic())
        if left <= 0: break
        await asyncio.wait({task}, timeout=left)
    return task.done()

async def shut_stream(ait, nxt=None, wait=CLOSE_WAIT):
    "Cancel the pending read `nxt`, close the async iterator `ait`, and return True when both end cleanly within `wait` seconds in total. A read or close that fails or runs past `wait` gives False. It never cancels a wait, because a cancelled close cuts short the SDK's own grace period, terminate and kill. A close that runs past `wait` keeps running."
    end = time.monotonic() + wait
    ok = True
    if nxt is not None:
        nxt.cancel()
        await asyncio.wait({nxt}, timeout=wait)
        ok = nxt.done() and (nxt.cancelled() or nxt.exception() is None)
    if ait is None: return ok
    close = asyncio.ensure_future(ait.aclose())
    close.add_done_callback(lambda t: t.cancelled() or t.exception())
    await asyncio.wait({close}, timeout=max(0, end - time.monotonic()))
    return ok and close.done() and not close.cancelled() and close.exception() is None

def split_model_path(model, model_path=None):
    "Split a model name, without its runtime prefix, into `(model_id, model_path)`: a Hub repo id or a local file path."
    model = split_runtime(model)[1]
    model_id = None if model is None or is_path(model) else model
    model_path = model_path or (model if model and is_path(model) else None)
    return model_id, model_path

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

## The shared layer

Backend modules import shared types such as `Chat`, `ChatOpts` and `Resp` from `urai`. Importing `rishi.core` registers the runtimes. It does not copy Urai's namespace.

`RishiToolLoop` is the base class for backends that drive Urai's tool loop. Llama.cpp, MLX, Ollama, the hosted API and Claude Code use it. Copilot uses it through the hosted API class. LiteRT-LM runs tools in its own engine. It does not use `RishiToolLoop`. `RishiToolLoop` enables three callbacks by default. `UsageCallback` counts token usage. `ToolReminderCallback` reminds the model about its tools. `SlidingWindowCallback` removes old messages when the context window is full.

`split_model_path` splits a model name into a Hugging Face repository id or a local file path. It removes a runtime prefix first. An explicit `model_path` takes priority over a path in the model name. LiteRT-LM, llama.cpp and MLX call it. Ollama does not call it.

In [ ]:
#| export
_all_ = ['BACKENDS', 'litert_caps', 'ollama_caps_', 'use_system_certs_', 'repo_root', 'mv_skill_md']

In [ ]:
import importlib, importlib.util, threading
from fastcore.aio import run_sync
import rishi, rishi.core, urai

test_eq(rishi.Chat is urai.Chat, True)
test_eq(hasattr(rishi.core, 'Chat'), False)
for mod in ('llama', 'mlx', 'remote', 'claude', 'copilot', 'ollama'):
    if mod == 'mlx' and not importlib.util.find_spec('mlx'): continue
    public = importlib.import_module(f'rishi.{mod}').__all__
    assert 'Chat' not in public and 'UsageStats' not in public

In [ ]:
test_eq(split_model_path(None), (None, None))
test_eq(split_model_path('mlx-community/Qwen3-4B-4bit'), ('mlx-community/Qwen3-4B-4bit', None))
test_eq(split_model_path('mlx/mlx-community/Qwen3-4B-4bit'), ('mlx-community/Qwen3-4B-4bit', None))
test_eq(split_model_path('model.gguf'), (None, 'model.gguf'))
test_eq(split_model_path('repo/id', 'given/path.gguf'), ('repo/id', 'given/path.gguf'))

`OneShotCancelled` is raised when `cancel()` ends `oneshot` or `oneshot_stream`. It applies when cancellation occurs before the first chunk or after a later chunk. Text yielded before cancellation is a partial reply. A cancelled `oneshot` returns no reply. `stopped` is `False` when the provider call did not end within `CLOSE_WAIT` seconds. The call can continue and spend tokens. `shut_stream` closes a stream within that limit. It does not cancel the close. Claude Code's SDK shields its close from `anyio` cancellation. It does not shield the close from `asyncio` cancellation. A cancelled close skips termination and killing.

In [ ]:
done = []
async def _slow_close():
    try: yield 1; await asyncio.sleep(10)
    finally: await asyncio.sleep(.4); done.append('closed')
async def _probe():
    ait = _slow_close().__aiter__()
    await ait.__anext__()
    t = time.monotonic()
    ended = await shut_stream(ait, wait=.1)
    early = time.monotonic() - t < .3
    await asyncio.sleep(.6)
    return ended, early, done[:]
test_eq(run_sync(_probe()), (False, True, ['closed']))

async def _poll():
    flag = threading.Event(); t = asyncio.ensure_future(asyncio.sleep(10))
    flag.set()
    r = await read_or_cancel(t, flag)
    pending = not t.done(); t.cancel()
    return r, pending
test_eq(run_sync(_poll()), (False, True))

## A SafePyRun patch

Urai runs model-generated Python through SafePyRun. SafePyRun through version 0.2.6 uses separate global and local namespaces for each code block. A function in the block cannot see a module imported in that block. For example, `import math` followed by a function that calls `math.sqrt` raises `NameError`. `rishi` replaces the SafePyRun runner with one shared namespace. Remove the replacement after SafePyRun fixes this behavior.

In [ ]:
#| export
import safepyrun.core as _spc

async def _rishi_run_python(code, g=None, ok_dests=None):
    _spc._rp_globals.set(g)
    inject = {'__builtins__': _spc._builtins, '__name__': '<python>'}
    protect = list(inject) + list(_spc._noexport)
    saved = {k: g[k] for k in protect if k in g}; had = set(saved)
    g.update(inject)
    async def run(src, is_exec=True):
        comp = compile(src, _spc.srcfn(src), 'exec' if is_exec else 'eval', flags=ast.PyCF_ALLOW_TOP_LEVEL_AWAIT)
        r = eval(comp, g)
        return (await r) if inspect.isawaitable(r) else r
    try:
        tree = ast.parse(code)
        warnings.filterwarnings('ignore', category=SyntaxWarning)
        res = None
        if tree.body and isinstance(tree.body[-1], ast.Expr):
            last = tree.body.pop()
            if tree.body: await run(ast.unparse(ast.Module(tree.body, [])))
            res = await run(ast.unparse(ast.Expression(last.value)), False)
        else: await run(code)
        return res
    finally:
        for k in protect:
            if k in had: g[k] = saved[k]
            else: g.pop(k, None)

setattr(_spc, '__run_python', _rishi_run_python)

The host's `__name__` and `__builtins__` remain unchanged when a block raises or fails to parse:

In [ ]:
def _drive(coro):
    try: coro.send(None)
    except StopIteration as e: return e.value

host = {'__name__': 'host', '__builtins__': {}}
for bad in ['raise ValueError', 'def (']: test_fail(lambda: _drive(_rishi_run_python(bad, host)))
test_eq(host['__builtins__'], {})
host['__name__']

## Model capabilities

A `Caps` record lists the input and output types that a model supports. Check it before sending an image to a model. Urai reads the `Caps` record for a hosted model from the fastllm model table. Local models have no such table. Each local runtime supplies a function that inspects its model. `litert_caps` reports text, image and audio for every LiteRT-LM model. LiteRT-LM builds vision and audio encoders when multimodal input is enabled. `ollama_caps_` asks the Ollama daemon that holds the model weights.

In [ ]:
#| export
def litert_caps(model=None, model_path=None):
    "Return text, image and audio input capabilities for a LiteRT-LM model."
    return Caps(('text', 'image', 'audio'), ('text',), (), 'runtime')

def ollama_caps_(model=None, model_path=None):
    "Return the capabilities reported by the Ollama daemon, or `None` when the backend is unavailable."
    try: from rishi.ollama import ollama_caps
    except ImportError: return None
    return ollama_caps(model)

In [ ]:
test_eq(litert_caps('anything').inp, ('text', 'image', 'audio'))
test_eq(litert_caps('anything').source, 'runtime')

## The seven backends

`BACKENDS` holds one `Runtime` record for each backend. A record contains the backend name, its implementation class, a capabilities function and the patterns that identify its models. Urai imports the class when the backend is first used. When `Chat` receives a model name without a runtime, Urai checks the patterns in list order and uses the first match. Ollama comes before llama.cpp. The id `hf.co/unsloth/Qwen3-4B-GGUF` contains `gguf`. It is an Ollama id, not a llama.cpp file. LiteRT-LM is the default when no model name is given.

Claude Code and Copilot have no patterns. Their model names look like hosted model names. Select them with a `claude/` or `copilot/` prefix. You can also set `runtime='claude'` or `runtime='copilot'`. A name that matches no pattern raises an error.

In [ ]:
#| export
#: Every runtime rishi serves, in the order `infer_runtime` should try them.
BACKENDS = [
    Runtime('litert', 'rishi.litert.LitertChat', optional=True, caps=litert_caps,
            pats=('.litertlm', 'litertlm', 'litert-community', 'litert-lm')),
    Runtime('ollama', 'rishi.ollama.OllamaChat', optional=True, caps=ollama_caps_,
            pats=('hf.co/', 'huggingface.co/')),
    Runtime('llama',  'rishi.llama.LlamaChat',   optional=True, caps=mmproj_caps,
            pats=('.gguf', 'gguf')),
    Runtime('mlx',    'rishi.mlx.MlxChat',       optional=True, caps=cfg_caps,
            pats=('mlx-community', 'mlx_lm', '-mlx', 'mlx-')),
    Runtime('remote', 'rishi.remote.RemoteChat', local=False,
            pats=('claude-', 'gpt-', 'gemini-', 'kimi-', 'deepseek-', 'grok-',
                  'sonnet', 'opus', 'haiku', 'fable')),
    Runtime('claude',  'rishi.claude.ClaudeChat',  optional=True, local=False),
    Runtime('copilot', 'rishi.copilot.CopilotChat', local=False),
]

for _rt in BACKENDS: register_runtime(_rt, dflt=_rt.name == 'litert')

In [ ]:
test_eq(list(RUNTIMES), ['litert', 'ollama', 'llama', 'mlx', 'remote', 'claude', 'copilot'])
test_eq(infer_runtime('gemma-3n-E4B.litertlm'), 'litert')
test_eq(infer_runtime('hf.co/unsloth/Qwen3-4B-GGUF'), 'ollama')
test_eq(infer_runtime('qwen3-4b-Q4_K_M.gguf'), 'llama')
test_eq(infer_runtime('mlx-community/Qwen3-4B-4bit'), 'mlx')
test_eq(infer_runtime('gpt-5.1'), 'remote')

In [ ]:
test_eq(resolve_runtime('claude-sonnet-4.6'), ('remote', 'claude-sonnet-4.6'))
test_eq(resolve_runtime('claude/claude-sonnet-4.6'), ('claude', 'claude-sonnet-4.6'))
test_eq(resolve_runtime()[0], 'litert')                  # nothing named: the default runtime
test_fail(lambda: resolve_runtime('some-unknown-name'), contains="Can't tell which backend")

In [ ]:
test_eq(model_caps('gemma-3n-E4B.litertlm').source, 'runtime')
test_eq({'text', 'image'} - set(model_caps('gpt-5.1').inp), set())   # fastllm's table grows, e.g. 'pdf'
test_eq(resolve('gpt-5.1').local, False)
test_eq(resolve('anything.litertlm').local, True)

## Declared dependencies

A plain `pip install rishi` must load every required runtime. Every package that a required runtime imports must be in `pyproject.toml` or in the dependency tree of a declared package. Before version 0.1.68, the `remote` and `copilot` backends imported `fastspec` without declaring it. Other packages often installed `fastspec`. The error appeared in projects that installed only `rishi`.

The test reads each runtime module and each imported `rishi` module. It collects their imported package names. It ignores the standard library. It also ignores imports in a `try` block that catches `ImportError`. It compares the result with the declared dependencies and their required dependencies.

In [ ]:
import ast, re, sys, tomllib, importlib.metadata as im

In [ ]:
def _norm(n): return re.sub(r'[-_.]+', '-', n).lower()
def _req_name(s): return _norm(re.match(r'[\w.\-]+', s)[0])

def _guarded(tree):
    "Return nodes in `try` blocks that catch `ImportError`."
    tries = [t for t in ast.walk(tree) if isinstance(t, ast.Try)
             and any(isinstance(h.type, ast.Name) and h.type.id in ('ImportError', 'ModuleNotFoundError') for h in t.handlers)]
    return {id(n) for t in tries for b in t.body for n in ast.walk(b)}

def _mod_imports(mod, seen=None):
    "Return the top-level packages that `mod` needs, including imported `rishi` modules."
    seen = set() if seen is None else seen
    if mod in seen: return set()
    seen.add(mod)
    tree = ast.parse(Path(importlib.util.find_spec(mod).origin).read_text())
    skip,res = _guarded(tree),set()
    for n in ast.walk(tree):
        if id(n) in skip: continue
        if isinstance(n, ast.Import): names = [a.name for a in n.names]
        elif isinstance(n, ast.ImportFrom): names = [f'rishi.{n.module}' if n.level else n.module]
        else: continue
        for o in names:
            if o.startswith('rishi.'): res |= _mod_imports(o, seen)
            elif o != 'rishi': res.add(o.split('.')[0])
    return res - set(sys.stdlib_module_names)

def _dep_closure(reqs):
    "Return distributions that installing `reqs` always brings, excluding extras."
    todo,res = [_req_name(r) for r in reqs],set()
    while todo:
        if (d := todo.pop()) in res: continue
        res.add(d)
        try: todo += [_req_name(r) for r in im.requires(d) or [] if 'extra ==' not in r]
        except im.PackageNotFoundError: pass
    return res

proj = tomllib.loads(Path('../pyproject.toml').read_text())['project']
pkg2dist = im.packages_distributions()
def _undeclared(mods, deps): return sorted(m for m in mods if not {_norm(d) for d in pkg2dist.get(m, [m])} & deps)

The `remote` backend needs `fastspec`. The `copilot` backend gets it through `fastspec`. A required runtime must use only the core dependencies:

In [ ]:
test_eq('fastspec' in _mod_imports('rishi.copilot'), True)
core = _dep_closure(proj['dependencies'])
test_eq({k: _undeclared(_mod_imports(rt.cls.rsplit('.', 1)[0]), core) for k,rt in RUNTIMES.items() if not rt.optional},
        {'remote': [], 'copilot': []})

A plain install also loads `ollama`. The `litert`, `llama`, `mlx` and `claude` runtimes need their extras. Each extra installs every package that its runtime imports. Only vision and audio MLX models load `mlx_vlm` from the `mlx-vlm` extra. The check skips an extra whose platform markers exclude this machine:

In [ ]:
def _rt_mods(rt): return _mod_imports(rt.cls.rsplit('.', 1)[0])
test_eq([k for k,rt in RUNTIMES.items() if _undeclared(_rt_mods(rt), core)], ['litert', 'llama', 'mlx', 'claude'])
from packaging.requirements import Requirement
extras = proj['optional-dependencies']
def _applies(reqs): return all(not (m := Requirement(r).marker) or m.evaluate({'extra': ''}) for r in reqs)
missing = {k: _undeclared(_rt_mods(rt), core | _dep_closure(extras[rt.extra or k])) for k,rt in RUNTIMES.items()
           if rt.optional and _applies(extras[rt.extra or k])}
test_eq({k: v for k,v in missing.items() if v}, {'mlx': ['mlx_vlm']} if 'mlx' in missing else {})

## Compatibility names

Some functions moved from `rishi` to Urai and have new names. The old names remain as aliases. For example, `mk_oai_msg` aliases `mk_msg`, and `browser_approval` aliases `http_approval`. Use Urai names in new code.

In [ ]:
#| export
#: Compatibility aliases for names Rishi used before Urai.
mk_oai_content, mk_oai_msg, mk_oai_msgs = mk_content, mk_msg, mk_msgs
browser_approval = http_approval
runtimes = RUNTIMES
_is_path = is_path

def use_system_certs_(force=False):
    "Deprecated alias for `urai.use_system_certs`."
    return use_system_certs(force)

In [ ]:
test_eq(mk_oai_msg('hi'), {'role': 'user', 'content': 'hi'})
test_eq(mk_oai_content('hi'), {'type': 'text', 'text': 'hi'})
test_eq(runtimes is RUNTIMES, True)
test_eq(_is_path('x.gguf'), True)

## Installing the skill

The wheel includes `skill.md`. The file tells a coding agent how to use `rishi`. `mv_skill_md` copies it to `.agents/skills/rishi/SKILL.md` and `.claude/skills/rishi/SKILL.md`. The `dir` argument sets the target root. Without `dir`, `repo_root` finds the current Git repository. Outside a repository, the target root is the current folder. `repo_root` returns `None` outside a repository. By default, `mv_skill_md` prints the target paths and copies nothing. Set `dry_run=False` to copy. If `skill.md` is missing, `mv_skill_md` returns without output.

In [ ]:
#| export
def repo_root() -> Path:
    "Root of the current git repository, or None if not in one."
    return find_file_parents('.git')

def mv_skill_md(dry_run=True, dir=None):
    "Copy the bundled `skill.md` into the repository's `.agents` and `.claude` skill directories, or print the target paths when `dry_run` is `True`."
    base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    if not (src := first((base/'skill.md', base.parent/'skill.md'), Path.exists)): return
    root = Path(dir or repo_root() or '.')
    ts = [root/'.agents/skills/rishi/SKILL.md', root/'.claude/skills/rishi/SKILL.md']
    if dry_run: print(f'Would copy {src} to: {list(map(str, ts))}')
    else:
        for p in ts: p.mk_write(src.read_text(encoding='utf-8'))
        print(f'Installed -> {list(map(str, ts))}')

In [ ]:
assert repo_root() is None or (repo_root()/'.git').exists()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()